Universidad del Valle de Guatemala  
Departamento de Ciencias de la Computación  
CC3085 - Inteligencia Artificial - sección 10  

Cristian Túnchez - 231359  
Nadissa Vela - 23764

# Laboratorio 10

## Task 2 - Depuración de un Sistema con Bug Deliberado

In [3]:
import numpy as np

CARRILES = 20
K = 10

def transicion(h_prev):
    # El vehiculo se mueve +1, 0 o -1 carril con igual probabilidad
    delta = np.random.choice([-1, 0, 1])
    return int(np.clip(h_prev + delta, 0, CARRILES - 1))
    
def emision(sensor, h):
    # Sensor reporta el carril real con prob 0.6, adyacente con 0.2, error con 0.2
    dist = abs(sensor - h)
    if dist == 0: return 0.6
    elif dist == 1: return 0.2
    else: return 0.2 / (CARRILES - 2)

def filtrado_particulas(observaciones):
    particulas = np.random.randint(0, CARRILES, K)
    for t, sensor in enumerate(observaciones):
        # PASO 1: Proponer
        propuestas = np.array([transicion(h) for h in particulas])
        # PASO 2: Ponderar
        pesos = np.array([emision(sensor, h) for h in propuestas])
        pesos_norm = pesos / pesos.sum()
        # PASO 3: Remuestrear <-- REVISEN ESTA LINEA
        idx = np.argsort(pesos_norm)[-K:]
        particulas = propuestas[idx]
        print(f"t={t+1} | sensor={sensor} | particulas={sorted(particulas)}")
    return particulas

# Secuencia de sensores simulando movimiento real del vehiculo
observaciones = [5, 6, 7, 7, 8, 8, 3, 4, 5]
filtrado_particulas(observaciones)

t=1 | sensor=5 | particulas=[np.int64(2), np.int64(4), np.int64(6), np.int64(6), np.int64(9), np.int64(11), np.int64(15), np.int64(15), np.int64(16), np.int64(18)]
t=2 | sensor=6 | particulas=[np.int64(3), np.int64(3), np.int64(5), np.int64(7), np.int64(10), np.int64(10), np.int64(16), np.int64(16), np.int64(16), np.int64(17)]
t=3 | sensor=7 | particulas=[np.int64(4), np.int64(4), np.int64(5), np.int64(6), np.int64(9), np.int64(10), np.int64(16), np.int64(16), np.int64(17), np.int64(18)]
t=4 | sensor=7 | particulas=[np.int64(3), np.int64(4), np.int64(5), np.int64(5), np.int64(8), np.int64(9), np.int64(15), np.int64(16), np.int64(18), np.int64(18)]
t=5 | sensor=8 | particulas=[np.int64(4), np.int64(4), np.int64(4), np.int64(5), np.int64(9), np.int64(9), np.int64(16), np.int64(17), np.int64(17), np.int64(19)]
t=6 | sensor=8 | particulas=[np.int64(3), np.int64(3), np.int64(4), np.int64(5), np.int64(9), np.int64(9), np.int64(16), np.int64(16), np.int64(17), np.int64(19)]
t=7 | sensor=3 | p

array([16, 19, 16, 17,  8, 10,  7,  2,  1,  4])

### Pregunta 1 — Identificación del Bug

**La línea con bug:**  

```python
idx = np.argsort(pesos_norm)[-K:]
```

**¿Por qué esto es Beam Search y no Filtrado de Partículas?**

`np.argsort(pesos_norm)[-K:]` selecciona de forma **determinista** los K índices con mayor peso. Eso es exactamente la definición de **Beam Search**: mantener en cada paso las K hipótesis con mayor puntuación, descartando todas las demás sin ningún componente aleatorio.

El Filtrado de Partículas exige que el remuestreo sea un **muestreo aleatorio con reemplazo** proporcional a los pesos normalizados, de modo que la distribución empírica de partículas aproxime la distribución posterior verdadera $ P(h_t | o_{1:t}) $.

**Propiedad matemática violada (esperanza proporcional al peso)**

En remuestreo correcto, la cantidad esperada de veces que se selecciona la partícula $i$ satisface:

$ E[count_i] = K · w_i $

Con Beam Search esa propiedad se rompe completamente:  

- Las K partículas con mayor peso reciben `count = 1`, independientemente de sus diferencias de peso relativas.
- Las restantes reciben `count = 0`, aunque tengan pesos no nulos.

Esto introduce un sesgo sistemático: el conjunto de partículas ya no es una muestra válida de la posterior, sino una selección greedy que colapsa la diversidad y sobre-representa regiones de alto peso.

### Pregunta 2 — Corrección del Bug y Comparación

**La corrección (exactamente una línea distinta):**

Buggy:  

```python
idx = np.argsort(pesos_norm)[-K:]
```

Corregida:  

```python
idx = np.random.choice(K, size=K, replace=True, p=pesos_norm)
```

`np.random.choice` muestrea $K$ índices **con reemplazo**, donde la probabilidad de seleccionar el índice $i$ es `pesos_norm[i]`. Esto restaura la propiedad $ E[count_i] = K · w_i $ y mantiene diversidad en el conjunto de partículas.

In [4]:
import numpy as np

CARRILES = 20
K = 10

def transicion(h_prev):
    delta = np.random.choice([-1, 0, 1])
    return int(np.clip(h_prev + delta, 0, CARRILES - 1))

def emision(sensor, h):
    dist = abs(sensor - h)
    if dist == 0: return 0.6
    elif dist == 1: return 0.2
    else: return 0.2 / (CARRILES - 2)

def filtrado_buggy(observaciones, seed=42):
    np.random.seed(seed)
    particulas = np.random.randint(0, CARRILES, K)
    resultados = []
    for t, sensor in enumerate(observaciones):
        propuestas = np.array([transicion(h) for h in particulas])
        pesos = np.array([emision(sensor, h) for h in propuestas])
        pesos_norm = pesos / pesos.sum()
        # BUG: Beam Search — top-K determinista
        idx = np.argsort(pesos_norm)[-K:]
        particulas = propuestas[idx]
        resultados.append((t+1, sensor, sorted(particulas.tolist())))
    return resultados

def filtrado_correcto(observaciones, seed=42):
    np.random.seed(seed)
    particulas = np.random.randint(0, CARRILES, K)
    resultados = []
    for t, sensor in enumerate(observaciones):
        propuestas = np.array([transicion(h) for h in particulas])
        pesos = np.array([emision(sensor, h) for h in propuestas])
        pesos_norm = pesos / pesos.sum()
        # CORRECCIÓN: muestreo aleatorio con reemplazo proporcional a pesos
        idx = np.random.choice(K, size=K, replace=True, p=pesos_norm)
        particulas = propuestas[idx]
        resultados.append((t+1, sensor, sorted(particulas.tolist())))
    return resultados

observaciones = [5, 6, 7, 7, 8, 8, 3, 4, 5]

res_bug = filtrado_buggy(observaciones, seed=42)
res_fix = filtrado_correcto(observaciones, seed=42)

print(f"{'t':>3} | {'sensor':>6} | {'BUGGY (Beam Search)':^30} | {'CORRECTO (Particle Filter)':^30}")
print("-" * 80)
for (t, s, bug_p), (_, _, fix_p) in zip(res_bug, res_fix):
    print(f"{t:>3} | {s:>6} | {str(bug_p):^30} | {str(fix_p):^30}")

  t | sensor |      BUGGY (Beam Search)       |   CORRECTO (Particle Filter)  
--------------------------------------------------------------------------------
  1 |      5 | [3, 6, 7, 7, 9, 9, 10, 13, 18, 19] | [6, 6, 6, 6, 6, 6, 6, 6, 6, 10]
  2 |      6 | [4, 6, 7, 8, 9, 9, 10, 12, 18, 19] | [5, 5, 5, 5, 6, 6, 6, 7, 7, 7]
  3 |      7 | [3, 7, 8, 9, 9, 10, 11, 13, 18, 18] | [6, 6, 6, 6, 6, 6, 6, 7, 7, 7]
  4 |      7 | [3, 7, 7, 9, 10, 10, 10, 12, 17, 18] | [7, 7, 7, 7, 7, 7, 7, 7, 7, 7]
  5 |      8 | [3, 6, 7, 9, 10, 11, 11, 12, 16, 19] | [7, 7, 7, 8, 8, 8, 8, 8, 8, 8]
  6 |      8 | [2, 7, 7, 9, 10, 11, 11, 12, 16, 19] | [7, 8, 8, 8, 8, 8, 8, 8, 8, 8]
  7 |      3 | [2, 7, 8, 8, 10, 11, 12, 12, 16, 19] | [7, 7, 7, 8, 8, 8, 8, 8, 8, 9]
  8 |      4 | [1, 6, 7, 7, 11, 11, 11, 12, 15, 18] | [6, 6, 6, 6, 6, 6, 7, 9, 9, 9]
  9 |      5 | [2, 6, 7, 8, 10, 10, 10, 11, 16, 19] | [5, 5, 5, 5, 5, 5, 6, 6, 6, 6]


### Pregunta 3 — Secuencia que expone el fallo del sistema con bug

**Secuencia diseñada (8 pasos):**
```
observaciones_fallo = [10, 10, 10, 18, 18, 18, 18, 18]
```

**Lógica de la secuencia:**

- **Pasos 1–3 (sensor = 10):** Señal fuerte y constante en carril 10. Con Beam Search, los K pesos más altos corresponden a partículas cercanas a 10. Tras 3 pasos, **todas las partículas del sistema buggy colapsan al entorno del carril 10** (diversidad ≈ 0). El sistema correcto también concentra partículas ahí, pero el muestreo estocástico conserva algunas partículas dispersas.

- **Pasos 4–8 (sensor = 18):** El vehículo salta al carril 18, que está a 8 posiciones de distancia. Con Beam Search, todas las partículas están cerca de 10; para cada una de ellas `emision(18, h)` es uniformemente baja y casi igual, así que los pesos son casi uniformes. El top-K sigue siendo el mismo cluster alrededor de 10 (**el sistema buggy nunca migra hacia 18**). El sistema correcto, al muestrear aleatoriamente (incluso con pesos casi iguales), permite que las partículas se redistribuyan gradualmente hacia 18 y se recupere el tracking.

**¿Por qué esta secuencia activa específicamente la diferencia?**

La diferencia entre Beam Search y Particle Filter es máxima cuando los pesos son **casi uniformes** (ninguna partícula destaca). Beam Search selecciona un top-K arbitrario pero estable (sin movimiento real), mientras que el muestreo aleatorio genera diversidad que permite explorar el espacio de estados. La transición abrupta de 10 a 18 fuerza ese régimen de pesos casi planos.

In [5]:
import numpy as np

# Secuencia diseñada para exponer el fallo: convergencia a 10, luego salto a 18
observaciones_fallo = [10, 10, 10, 18, 18, 18, 18, 18]

res_bug2 = filtrado_buggy(observaciones_fallo, seed=42)
res_fix2 = filtrado_correcto(observaciones_fallo, seed=42)

print("Secuencia: convergencia en carril 10, luego salto abrupto a carril 18")
print(f"\n{'t':>3} | {'sensor':>6} | {'BUGGY — colapsa, no recupera':^32} | {'CORRECTO — se recupera':^32}")
print("-" * 85)
for (t, s, bug_p), (_, _, fix_p) in zip(res_bug2, res_fix2):
    marker = " <-- SALTO" if s == 18 and t == 4 else ""
    print(f"{t:>3} | {s:>6} | {str(bug_p):^32} | {str(fix_p):^32}{marker}")

# Mostrar mediana de partículas para ver la diferencia de seguimiento
print("\nMediana de partículas por paso:")
print(f"{'t':>3} | {'sensor':>6} | {'mediana buggy':>14} | {'mediana correcto':>16} | {'error buggy':>12} | {'error correcto':>14}")
print("-" * 75)
for (t, s, bug_p), (_, _, fix_p) in zip(res_bug2, res_fix2):
    mb = np.median(bug_p)
    mf = np.median(fix_p)
    print(f"{t:>3} | {s:>6} | {mb:>14.1f} | {mf:>16.1f} | {abs(mb-s):>12.1f} | {abs(mf-s):>14.1f}")

Secuencia: convergencia en carril 10, luego salto abrupto a carril 18

  t | sensor |   BUGGY — colapsa, no recupera   |      CORRECTO — se recupera     
-------------------------------------------------------------------------------------
  1 |     10 | [3, 6, 7, 7, 9, 9, 10, 13, 18, 19] | [9, 10, 10, 10, 10, 10, 10, 10, 10, 18]
  2 |     10 | [4, 6, 7, 7, 9, 10, 10, 12, 18, 19] | [9, 9, 9, 9, 10, 10, 10, 10, 11, 11]
  3 |     10 | [5, 6, 7, 7, 8, 11, 11, 12, 19, 19] | [9, 9, 10, 10, 10, 10, 10, 10, 10, 10]
  4 |     18 | [5, 6, 6, 7, 8, 10, 11, 13, 18, 18] | [8, 8, 9, 9, 9, 9, 9, 9, 10, 11] <-- SALTO
  5 |     18 | [5, 6, 7, 7, 8, 10, 12, 13, 17, 18] | [7, 8, 8, 8, 9, 9, 9, 10, 10, 10]
  6 |     18 | [5, 6, 7, 7, 8, 11, 11, 13, 17, 18] | [9, 9, 9, 9, 9, 9, 9, 9, 10, 11]
  7 |     18 | [5, 7, 7, 8, 8, 11, 12, 13, 16, 18] | [8, 9, 9, 9, 9, 9, 9, 11, 11, 11]
  8 |     18 | [4, 7, 7, 8, 8, 10, 11, 12, 15, 17] | [8, 10, 10, 10, 10, 10, 10, 10, 10, 11]

Mediana de partículas por paso:
  t 

### Pregunta 4 — Condición bajo la cual ambas versiones producen resultados idénticos

**Condición: una sola partícula concentra prácticamente todo el peso ($w_i ≈ 1$, $resto ≈ 0$).**

Esto ocurre cuando exactamente una partícula propuesta coincide con el sensor (o está adyacente) y todas las demás están a distancia ≥ 2, de modo que:

```
pesos_norm ≈ [0, 0, ..., 1, ..., 0]   (un único peso dominante)
```

- **Beam Search** selecciona el top-K: el índice con $w ≈ 1$ es seleccionado K veces (ocupa los K primeros puestos de argsort).
- **Particle Filter** muestrea con reemplazo: la probabilidad de NO seleccionar ese índice en ninguno de los K sorteos es $(1 - 1)^K ≈ 0$. En la práctica también lo selecciona K veces.

Ambos métodos producen el mismo conjunto de partículas: K copias de la única partícula con peso significativo.

**Justificación matemática:**

Sea $p^* = max(pesos_{norm})$. Cuando $p^* → 1$:  

- Beam Search: selecciona el índice $i^*$ con certeza.
- Multinomial: $P(i^*$ seleccionado en los K sorteos $) = 1 - (1 - p^*)^K → 1$.

El límite es idéntico. En la práctica, con `K = 10` y $p^* ≥ 0.99$, la probabilidad de que el muestreo aleatorio no seleccione $i^*$ al menos una vez es $< 10^{-20}$, así que para todos los fines prácticos ambas versiones colapsan al mismo resultado.

**Escenario concreto en LogiTrack:** si hay 10 partículas en el carril 5 y el sensor reporta 5, todas las propuestas cercanas a 5 tienen alta probabilidad, pero si solo una cae exactamente en 5 y las otras 9 están en carriles distantes ($≥ 7$), se alcanza la condición descrita.